In [ ]:
# [Setup]

# In VSCodium :
    # Ctrl + Shift + P -> Python: Select Interpreter -> Python 3.11 (rcbplates)

In [1]:
# ============================================================
# COMPLETE V CrA PHOTOMETRY PIPELINE - FIXED VERSION
# ============================================================

from pathlib import Path
from astropy.io import fits as astrofits
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder, find_peaks
from photutils.aperture import CircularAperture, CircularAnnulus, ApertureStats, aperture_photometry
from photutils.centroids import centroid_sources, centroid_com
from photutils.profiles import RadialProfile
from photutils.background import Background2D, MedianBackground
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy.time import Time
import astropy.units as u
from astroquery.simbad import Simbad
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
from scipy import ndimage, optimize
from skimage.transform import hough_line, hough_line_peaks
import pandas as pd
import pickle
import time
import urllib.request
import urllib.parse
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display, clear_output
from threading import Thread
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONSTANTS & VERSIONS
# ============================================================
VERSION = 3  # Bump to force reprocessing

TARGET_NAME = "V CrA"
TARGET_COORD = SkyCoord(ra=281.884623417 * u.deg, dec=-38.158974417 * u.deg)

# Photometry parameters
APERTURE_RADII = [15, 25, 40, 60]
DEFAULT_APERTURE = 25
ANNULUS_INNER = 45
ANNULUS_OUTER = 60
CENTROID_BOX = 21
PSF_FWHM = 3.0

# Detection thresholds
SIGNIF_THRESHOLD = 3.0
SATURATION_FRACTION = 0.995
SATURATION_MIN_PIXELS = 5
PLATE_LIMIT_NSIGMA = 3.0

# Matching tolerances
TARGET_MATCH_ARCSEC = 5.0
TARGET_MATCH_WIDENED_ARCSEC = 8.0
PARADIGM_MATCH_ARCSEC = 5.0
GAIA_MATCH_ARCSEC = 2.0

# Quality thresholds
MIN_APASS_STARS = 10
MAX_FIT_RMS = 0.5
MIN_SNR = 5.0
MIN_FIT_SLOPE = 0.3

# Review
REVIEW_MAX_TOTAL = 20
MAX_DROPDOWN_OPTIONS = 500

# ============================================================
# PATHS & CACHE
# ============================================================
cutout_dir = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts")
manifest_path = cutout_dir.parent / "plate_manifest.csv"
cache_dir = cutout_dir.parent / "02_A_fixed"
cache_dir.mkdir(parents=True, exist_ok=True)

cutouts = sorted(cutout_dir.glob("*.fits"))
print(f"Found {len(cutouts)} cutouts")

def load_cache(name, default=None):
    path = cache_dir / f"{name}.pkl"
    if path.exists():
        try:
            with open(path, 'rb') as f:
                return pickle.load(f)
        except:
            pass
    return default if default is not None else {}

def save_cache(name, data):
    with open(cache_dir / f"{name}.pkl", 'wb') as f:
        pickle.dump(data, f)

# Load all caches
plate_db = load_cache('plate_db', {})
apass_cache = load_cache('apass_cache', {})
gaia_cache = load_cache('gaia_cache', {})
simbad_cache = load_cache('simbad_cache', {})
paradigm_db = load_cache('paradigm_db', {})
review_db = load_cache('review_db', {})

print(f"Loaded {len(plate_db)} cached plates")

# ============================================================
# PLATE MANIFEST LOADING (FIXED)
# ============================================================
plate_limit_lookup = {}
if manifest_path.exists():
    try:
        manifest_df = pd.read_csv(manifest_path)
        print(f"Loaded manifest: {len(manifest_df)} rows")
        
        if 'filename' in manifest_df.columns:
            for _, row in manifest_df.iterrows():
                fname = row.get('filename')
                # Skip if filename is not a valid string
                if not isinstance(fname, str) or pd.isna(fname):
                    continue
                
                # Safely get lim_mag values
                lim_apass = row.get('lim_mag_apass')
                lim_atlas = row.get('lim_mag_atlas')
                
                # Convert to float if not NaN
                try:
                    lim_apass_val = float(lim_apass) if pd.notna(lim_apass) and lim_apass is not None else None
                except (ValueError, TypeError):
                    lim_apass_val = None
                
                try:
                    lim_atlas_val = float(lim_atlas) if pd.notna(lim_atlas) and lim_atlas is not None else None
                except (ValueError, TypeError):
                    lim_atlas_val = None
                
                plate_limit_lookup[Path(fname).name] = {
                    'lim_mag_apass': lim_apass_val,
                    'lim_mag_atlas': lim_atlas_val,
                }
            
            n_with_limits = sum(1 for v in plate_limit_lookup.values() 
                               if v['lim_mag_apass'] is not None or v['lim_mag_atlas'] is not None)
            print(f"Loaded limits for {n_with_limits} of {len(plate_limit_lookup)} plates")
        else:
            print("Manifest missing 'filename' column")
    except Exception as e:
        print(f"Error loading manifest: {e}")
else:
    print("No plate_manifest.csv found")

def get_plate_limits(fits_path):
    """Get plate limits from manifest."""
    entry = plate_limit_lookup.get(fits_path.name)
    if entry:
        return entry.get('lim_mag_apass'), entry.get('lim_mag_atlas')
    return None, None

# ============================================================
# CATALOG QUERIES (Cached)
# ============================================================
def query_vizier(source, ra, dec, radius_arcsec, columns, max_rows=5000):
    """Query Vizier with retries and caching."""
    params = {
        '-source': source,
        '-c': f"{ra} {dec}",
        '-c.rs': radius_arcsec,
        '-out': columns,
        '-out.max': max_rows,
    }
    url = "https://vizier.cds.unistra.fr/viz-bin/asu-tsv?" + urllib.parse.urlencode(params)
    
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=30) as response:
                return response.read().decode('utf-8')
        except:
            time.sleep(2**attempt)
    return None

def parse_vizier_tsv(text, converters):
    """Parse TSV with proper type conversion."""
    rows = []
    for line in text.split('\n'):
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('-'):
            continue
        parts = line.split('\t')
        if len(parts) < len(converters):
            continue
        try:
            row = tuple(conv(p) for conv, p in zip(converters, parts[:len(converters)]))
            rows.append(row)
        except:
            continue
    if rows and not isinstance(rows[0][0], (int, float)):
        rows = rows[1:]
    return rows

def get_field_catalog(wcs, shape, catalog_type='apass'):
    """Get catalog stars for the full field."""
    ra_c, dec_c = wcs.all_pix2world(shape[1]/2, shape[0]/2, 0)
    key = (round(ra_c, 3), round(dec_c, 3))
    
    if catalog_type == 'apass':
        cache = apass_cache
        source = 'II/336/apass9'
        columns = 'RAJ2000,DEJ2000,Bmag,e_Bmag,recno'
        converters = [float, float, float, float, int]
    else:  # gaia
        cache = gaia_cache
        source = 'I/355/gaiadr3'
        columns = 'RA_ICRS,DE_ICRS,Source,Plx,e_Plx,RV'
        converters = [float, float, int, float, float, float]
    
    if key in cache:
        return cache[key]
    
    # Compute field radius
    corners = [(0,0), (shape[1]-1,0), (0,shape[0]-1), (shape[1]-1,shape[0]-1)]
    corner_ra, corner_dec = wcs.all_pix2world([c[0] for c in corners], [c[1] for c in corners], 0)
    center = SkyCoord(ra_c*u.deg, dec_c*u.deg)
    corners_sky = SkyCoord(corner_ra*u.deg, corner_dec*u.deg)
    radius = center.separation(corners_sky).max().arcsec * 1.05
    
    try:
        raw = query_vizier(source, ra_c, dec_c, radius, columns)
        if raw:
            rows = parse_vizier_tsv(raw, converters)
            if catalog_type == 'apass':
                rows = [r for r in rows if r[2] > 0 and r[2] < 20 and r[3] < 0.5]
            cache[key] = rows
            save_cache('apass_cache' if catalog_type == 'apass' else 'gaia_cache', cache)
            print(f"{catalog_type.upper()}: {len(rows)} stars")
            return rows
    except Exception as e:
        print(f"{catalog_type.upper()} error: {e}")
    
    cache[key] = []
    return []

def get_simbad_catalog(wcs, shape):
    """Get SIMBAD objects in field."""
    ra_c, dec_c = wcs.all_pix2world(shape[1]/2, shape[0]/2, 0)
    key = (round(ra_c, 3), round(dec_c, 3))
    
    if key in simbad_cache:
        return simbad_cache[key]
    
    corners = [(0,0), (shape[1]-1,0), (0,shape[0]-1), (shape[1]-1,shape[0]-1)]
    corner_ra, corner_dec = wcs.all_pix2world([c[0] for c in corners], [c[1] for c in corners], 0)
    center = SkyCoord(ra_c*u.deg, dec_c*u.deg)
    corners_sky = SkyCoord(corner_ra*u.deg, corner_dec*u.deg)
    radius = center.separation(corners_sky).max().arcsec * 1.05
    
    try:
        simbad = Simbad()
        simbad.TIMEOUT = 30
        simbad.add_votable_fields('main_id', 'ra', 'dec')
        result = simbad.query_region(center, radius=radius*u.arcsec)
        simbad_cache[key] = result
        save_cache('simbad_cache', simbad_cache)
        print(f"SIMBAD: {len(result) if result is not None else 0} objects")
        return result
    except:
        simbad_cache[key] = []
        return []

# ============================================================
# IMAGE PROCESSING (Fixed)
# ============================================================
def subtract_background_2d(data, box_size=50):
    """2D background subtraction."""
    try:
        bkg_estimator = MedianBackground()
        bkg = Background2D(data, (box_size, box_size), 
                          filter_size=(3, 3), 
                          bkg_estimator=bkg_estimator)
        return data - bkg.background, bkg.background_rms
    except:
        return data - np.nanmedian(data), np.nanstd(data)

def measure_photometry(data, x, y, radii=APERTURE_RADII, 
                       annulus_inner=ANNULUS_INNER, 
                       annulus_outer=ANNULUS_OUTER):
    """Measure flux in multiple apertures with local background."""
    positions = list(zip(x, y))
    n = len(x)
    
    if n == 0:
        return {r: np.array([]) for r in radii}, np.array([]), np.array([])
    
    # Create apertures
    apertures = {r: CircularAperture(positions, r=r) for r in radii}
    annulus = CircularAnnulus(positions, r_in=annulus_inner, r_out=annulus_outer)
    
    # Get background
    bkg_median = np.zeros(n)
    bkg_std = np.zeros(n)
    
    for i in range(n):
        mask = annulus.to_mask(method='center')
        sl_large, sl_small = mask.get_overlap_slices(data.shape)
        if sl_large is not None:
            ann_data = data[sl_large][mask.data[sl_small] > 0]
            if len(ann_data) > 10:
                _, med, std = sigma_clipped_stats(ann_data, sigma=3.0)
                bkg_median[i] = med
                bkg_std[i] = std
    
    # Measure flux in each aperture
    fluxes = {}
    for r, ap in apertures.items():
        phot = aperture_photometry(data, ap)
        fluxes[r] = phot['aperture_sum'].value - bkg_median * (np.pi * r**2)
    
    return fluxes, bkg_median, bkg_std

def refine_centroids(data, x, y, box_size=CENTROID_BOX):
    """Centroid with convergence checking."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    
    if len(x) == 0:
        return x, y
    
    try:
        x_ref, y_ref = centroid_sources(data, x, y, 
                                       box_size=box_size,
                                       centroid_func=centroid_com)
        
        # Verify centroids
        good = np.isfinite(x_ref) & np.isfinite(y_ref)
        
        for i in range(len(x)):
            if not good[i]:
                continue
            x0, y0 = int(round(x_ref[i])), int(round(y_ref[i]))
            
            # Check bounds
            if x0 < 2 or x0 >= data.shape[1]-2 or y0 < 2 or y0 >= data.shape[0]-2:
                good[i] = False
                continue
            
            # Check if local peak
            patch = data[y0-1:y0+2, x0-1:x0+2]
            if patch.max() <= patch[1,1] or patch[1,1] <= np.nanmedian(data):
                good[i] = False
        
        x_ref[~good] = x[~good]
        y_ref[~good] = y[~good]
        return x_ref, y_ref
        
    except:
        return x, y

def compute_aperture_correction(data, x, y, apertures):
    """Compute aperture corrections using bright, isolated stars."""
    if len(x) < 3:
        return {r: 0.0 for r in apertures}
    
    # Find isolated stars
    isolation = np.full(len(x), np.inf)
    for i in range(len(x)):
        d = np.hypot(x - x[i], y - y[i])
        d[i] = np.inf
        isolation[i] = d.min()
    
    isolated = isolation > 40  # pixels
    if isolated.sum() < 3:
        return {r: 0.0 for r in apertures}
    
    # Use median aperture correction
    corrections = {}
    radii = sorted(apertures.keys())
    base_r = radii[0]
    
    for r in radii[1:]:
        ratio = apertures[r][isolated] / apertures[base_r][isolated]
        ratio = ratio[np.isfinite(ratio) & (ratio > 0)]
        if len(ratio) > 0:
            corrections[r] = -2.5 * np.log10(np.nanmedian(ratio))
        else:
            corrections[r] = 0.0
    
    return corrections

def fit_psf_fwhm(data, x, y, max_radius=30):
    """Fit 2D Gaussian to get FWHM."""
    fwhm = np.full(len(x), np.nan)
    
    for i in range(len(x)):
        try:
            x0, y0 = int(round(x[i])), int(round(y[i]))
            r = min(max_radius, min(x0, y0, data.shape[1]-x0, data.shape[0]-y0))
            if r < 5:
                continue
            
            x1, x2 = x0-r, x0+r+1
            y1, y2 = y0-r, y0+r+1
            sub = data[y1:y2, x1:x2]
            
            yy, xx = np.meshgrid(np.arange(sub.shape[0]), np.arange(sub.shape[1]), indexing='ij')
            total = sub.sum()
            if total <= 0:
                continue
            cx = (xx * sub).sum() / total
            cy = (yy * sub).sum() / total
            
            def gaussian(params):
                amp, x0, y0, sigx, sigy, bg = params
                model = bg + amp * np.exp(-((xx-x0)**2/(2*sigx**2) + (yy-y0)**2/(2*sigy**2)))
                return (model - sub).ravel()
            
            p0 = [sub.max(), cx, cy, r/3, r/3, np.nanmedian(sub)]
            result = optimize.least_squares(gaussian, p0, 
                                           bounds=([0,0,0,0.5,0.5,0], [np.inf,r*2,r*2,r*2,r*2,np.inf]))
            sigx, sigy = abs(result.x[3]), abs(result.x[4])
            
            if sigx > 0.5 and sigy > 0.5:
                fwhm[i] = 2.355 * np.sqrt((sigx**2 + sigy**2) / 2)
                
        except:
            continue
    
    return fwhm

def detect_saturation(data, x, y, apertures, sat_fraction=SATURATION_FRACTION):
    """Detect saturated pixels."""
    is_saturated = np.zeros(len(x), dtype=bool)
    max_vals = np.full(len(x), np.nan)
    
    for i in range(len(x)):
        mask = apertures[DEFAULT_APERTURE].to_mask(method='center')
        cut = mask.multiply(data)
        if cut is None:
            continue
        star_pixels = cut[mask.data > 0]
        if len(star_pixels) == 0:
            continue
        max_val = float(star_pixels.max())
        max_vals[i] = max_val
        near_max = np.sum(star_pixels >= max_val * sat_fraction)
        is_saturated[i] = near_max > SATURATION_MIN_PIXELS
    
    return is_saturated, max_vals

# ============================================================
# CALIBRATION (Fixed)
# ============================================================
def calibrate_photometry(inst_mags, apass_b, apass_err, 
                         outlier_sigma=3.0, min_stars=MIN_APASS_STARS):
    """
    Calibrate with outlier rejection and proper uncertainty.
    """
    mask = np.isfinite(inst_mags) & np.isfinite(apass_b) & (apass_b > 0) & (apass_b < 20)
    mask &= np.isfinite(inst_mags) & (inst_mags > -10) & (inst_mags < 30)
    
    if mask.sum() < min_stars:
        return None
    
    x = apass_b[mask]
    y = inst_mags[mask]
    
    # Iterative outlier rejection
    for iteration in range(3):
        try:
            coeffs = np.polyfit(x, y, 1)
            residuals = y - (coeffs[0]*x + coeffs[1])
            rms = np.sqrt(np.mean(residuals**2))
            
            good = np.abs(residuals) < outlier_sigma * rms
            if good.sum() < min_stars:
                break
            x = x[good]
            y = y[good]
        except:
            break
    
    if len(x) < min_stars:
        return None
    
    # Final fit with errors
    coeffs = np.polyfit(x, y, 1)
    residuals = y - (coeffs[0]*x + coeffs[1])
    rms = np.sqrt(np.mean(residuals**2))
    
    # Uncertainty estimates
    n = len(x)
    x_mean = np.mean(x)
    x_std = np.std(x)
    slope_err = rms / (np.sqrt(n) * x_std) if x_std > 0 else np.inf
    intercept_err = rms * np.sqrt(1/n + x_mean**2/(n * x_std**2)) if x_std > 0 else np.inf
    
    return {
        'slope': coeffs[0],
        'intercept': coeffs[1],
        'rms': rms,
        'n_used': n,
        'slope_err': slope_err,
        'intercept_err': intercept_err,
        'outliers_removed': mask.sum() - n
    }

def compute_upper_limit(flux, bkg_std, aperture_area, n_sigma=PLATE_LIMIT_NSIGMA):
    """Compute proper n-sigma upper limit."""
    noise = n_sigma * bkg_std * np.sqrt(aperture_area)
    if noise <= 0:
        return np.nan
    return -2.5 * np.log10(max(flux, noise))

def compute_plate_limit(bkg_std, calibration, aperture_area=np.pi*DEFAULT_APERTURE**2):
    """Compute 5-sigma detection limit."""
    if calibration is None or len(bkg_std) == 0:
        return None
    
    valid_std = bkg_std[np.isfinite(bkg_std) & (bkg_std > 0)]
    if len(valid_std) == 0:
        return None
    
    mean_bkg_std = np.nanmedian(valid_std)
    if mean_bkg_std <= 0:
        return None
    
    threshold_flux = 5.0 * mean_bkg_std * np.sqrt(aperture_area)
    if threshold_flux <= 0:
        return None
    
    inst_mag = -2.5 * np.log10(threshold_flux)
    return (inst_mag - calibration['intercept']) / calibration['slope']

# ============================================================
# SOURCE DETECTION (Minimal false positives)
# ============================================================
def detect_sources_robust(data, wcs, apass_stars, target_coord):
    """
    Detect sources with minimal false positives.
    """
    if not apass_stars:
        return None, None, None, None
    
    # Convert catalog positions
    ra = np.array([s[0] for s in apass_stars])
    dec = np.array([s[1] for s in apass_stars])
    bmag = np.array([s[2] for s in apass_stars])
    bmag_err = np.array([s[3] for s in apass_stars])
    
    x, y = wcs.all_world2pix(ra, dec, 0)
    
    # Keep on-frame sources
    margin = 30
    in_frame = (x > margin) & (x < data.shape[1]-margin) & \
               (y > margin) & (y < data.shape[0]-margin)
    x, y = x[in_frame], y[in_frame]
    ra, dec = ra[in_frame], dec[in_frame]
    bmag, bmag_err = bmag[in_frame], bmag_err[in_frame]
    
    if len(x) == 0:
        return None, None, None, None
    
    # Refine centroids
    x_ref, y_ref = refine_centroids(data, x, y)
    
    # Measure photometry
    fluxes, bkg_med, bkg_std = measure_photometry(data, x_ref, y_ref)
    flux = fluxes[DEFAULT_APERTURE]
    
    # Calculate SNR
    area = np.pi * DEFAULT_APERTURE**2
    snr = flux / (bkg_std * np.sqrt(area))
    
    # Only keep significant detections with proper centroids
    significant = (snr >= SIGNIF_THRESHOLD) & np.isfinite(snr)
    significant &= np.isfinite(flux) & (flux > 0)
    
    x_det = x_ref[significant]
    y_det = y_ref[significant]
    ra_det = ra[significant]
    dec_det = dec[significant]
    bmag_det = bmag[significant]
    bmag_err_det = bmag_err[significant]
    flux_det = flux[significant]
    bkg_std_det = bkg_std[significant]
    snr_det = snr[significant]
    
    # Find target
    target_idx = None
    target_tier = None
    
    if len(ra_det) > 0:
        coords = SkyCoord(ra_det*u.deg, dec_det*u.deg)
        sep = target_coord.separation(coords)
        min_sep_idx = np.argmin(sep)
        
        if sep[min_sep_idx].arcsec < TARGET_MATCH_ARCSEC:
            target_idx = min_sep_idx
            target_tier = 'catalog'
        elif sep[min_sep_idx].arcsec < TARGET_MATCH_WIDENED_ARCSEC:
            target_idx = min_sep_idx
            target_tier = 'catalog_widened'
    
    # Direct search if not found
    if target_idx is None:
        tx, ty = wcs.world_to_pixel_values(target_coord.ra.deg, target_coord.dec.deg)
        if margin < tx < data.shape[1]-margin and margin < ty < data.shape[0]-margin:
            search_r = 20
            x0 = max(0, int(tx-search_r))
            x1 = min(data.shape[1], int(tx+search_r+1))
            y0 = max(0, int(ty-search_r))
            y1 = min(data.shape[0], int(ty+search_r+1))
            
            local = data[y0:y1, x0:x1]
            if local.size > 0 and local.max() > np.nanpercentile(data, 95):
                cy, cx = np.unravel_index(np.argmax(local), local.shape)
                tx_found = x0 + cx
                ty_found = y0 + cy
                
                # Append target
                x_det = np.append(x_det, tx_found)
                y_det = np.append(y_det, ty_found)
                ra_det = np.append(ra_det, float(wcs.all_pix2world(tx_found, ty_found, 0)[0]))
                dec_det = np.append(dec_det, float(wcs.all_pix2world(tx_found, ty_found, 0)[1]))
                bmag_det = np.append(bmag_det, np.nan)
                bmag_err_det = np.append(bmag_err_det, np.nan)
                flux_det = np.append(flux_det, local.max())
                bkg_std_det = np.append(bkg_std_det, np.nanstd(data))
                snr_det = np.append(snr_det, local.max() / np.nanstd(data))
                target_idx = len(x_det) - 1
                target_tier = 'direct'
    
    return {
        'x': x_det, 'y': y_det, 'ra': ra_det, 'dec': dec_det,
        'bmag': bmag_det, 'bmag_err': bmag_err_det,
        'flux': flux_det, 'bkg_std': bkg_std_det, 'snr': snr_det,
        'bkg_med': bkg_med, 'bkg_std_all': bkg_std
    }, target_idx, target_tier

# ============================================================
# DEFECT DETECTION (Preserved from original)
# ============================================================
def detect_plate_errors(data):
    """Full defect detection from original code."""
    errors = {
        'scratches': [],
        'trailing': [],
        'saturation': [],
        'dust': [],
        'edge': False,
        'dead_zone_fraction': 0.0,
        'saturation_area_fraction': 0.0,
    }
    
    h, w = data.shape
    lo, hi = np.nanpercentile(data, 1), np.nanpercentile(data, 99)
    if hi == lo:
        return errors
    norm = np.clip((data - lo) / (hi - lo), 0, 1)
    
    # Scratches
    try:
        scale = 6
        small = ndimage.zoom(norm, 1/scale, order=1)
        sobel_h = ndimage.sobel(small, axis=0)
        sobel_v = ndimage.sobel(small, axis=1)
        edges = np.hypot(sobel_h, sobel_v)
        edges = (edges > np.percentile(edges, 97)).astype(np.uint8)
        
        tested_angles = np.linspace(-np.pi/2, np.pi/2, 90, endpoint=False)
        hspace, angles, dists = hough_line(edges, theta=tested_angles)
        peaks = hough_line_peaks(hspace, angles, dists, num_peaks=8, 
                                min_distance=20, threshold=0.35*hspace.max())
        
        hspace_mean = float(np.mean(hspace))
        hspace_std = float(np.std(hspace))
        sig_thresh = hspace_mean + 6 * hspace_std
        
        sh, sw = small.shape
        for peak_val, angle, dist in zip(*peaks):
            if peak_val < sig_thresh:
                continue
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            if abs(sin_a) > 1e-6:
                x0_s, x1_s = 0, sw-1
                y0_s = (dist - x0_s*cos_a) / sin_a
                y1_s = (dist - x1_s*cos_a) / sin_a
            else:
                y0_s, y1_s = 0, sh-1
                x0_s = x1_s = dist/cos_a if abs(cos_a) > 1e-6 else 0
            
            errors['scratches'].append({
                'x0': float(np.clip(x0_s*scale, 0, w-1)),
                'y0': float(np.clip(y0_s*scale, 0, h-1)),
                'x1': float(np.clip(x1_s*scale, 0, w-1)),
                'y1': float(np.clip(y1_s*scale, 0, h-1)),
            })
    except:
        pass
    
    # Trailing stars
    try:
        bright_mask = (norm > np.percentile(norm, 95)).astype(np.uint8)
        labeled, n_obj = ndimage.label(bright_mask)
        
        for obj_id in range(1, n_obj+1):
            region = labeled == obj_id
            area = region.sum()
            if area < 40 or area > 0.005*h*w:
                continue
            
            coords = np.argwhere(region)
            if len(coords) < 10:
                continue
            
            cov = np.cov(coords[:,1], coords[:,0])
            eigs = np.linalg.eigvalsh(cov)
            eigs = np.sort(np.abs(eigs))
            if eigs[0] < 1e-6:
                continue
            
            elongation = np.sqrt(eigs[1]/eigs[0])
            if elongation > 5.0:
                cy_r, cx_r = coords.mean(axis=0)
                angle = 0.5 * np.degrees(np.arctan2(2*cov[0,1], cov[0,0]-cov[1,1]))
                errors['trailing'].append({
                    'x': float(cx_r), 'y': float(cy_r),
                    'w': float(2*np.sqrt(eigs[1])), 'h': float(2*np.sqrt(eigs[0])),
                    'angle': float(angle)
                })
    except:
        pass
    
    # Saturation
    try:
        sat_thresh = np.percentile(norm, 99.9)
        sat_mask = (norm >= sat_thresh).astype(np.uint8)
        sat_mask = ndimage.binary_dilation(sat_mask, iterations=2).astype(np.uint8)
        errors['saturation_area_fraction'] = float(sat_mask.sum()) / float(h*w)
        
        labeled, n_obj = ndimage.label(sat_mask)
        for obj_id in range(1, n_obj+1):
            region = labeled == obj_id
            area = region.sum()
            if area < 200:
                continue
            coords = np.argwhere(region)
            cy_r, cx_r = coords.mean(axis=0)
            ry = (coords[:,0].max() - coords[:,0].min()) / 2
            rx = (coords[:,1].max() - coords[:,1].min()) / 2
            errors['saturation'].append({
                'x': float(cx_r), 'y': float(cy_r), 'r': float(np.hypot(rx, ry))
            })
    except:
        pass
    
    # Dust
    try:
        bg_size = max(15, min(h,w)//12)
        local_bg = ndimage.uniform_filter(norm, size=bg_size)
        residual = norm - local_bg
        med_resid = float(np.nanmedian(residual))
        mad = float(np.nanmedian(np.abs(residual - med_resid)))
        sigma_est = max(1.4826*mad, 1e-3)
        
        dark_mask = (residual <= (med_resid - 5*sigma_est)).astype(np.uint8)
        dark_mask = ndimage.binary_opening(dark_mask, iterations=1).astype(np.uint8)
        labeled, n_obj = ndimage.label(dark_mask)
        
        for obj_id in range(1, n_obj+1):
            region = labeled == obj_id
            area = region.sum()
            if area < 15 or area > 0.02*h*w:
                continue
            coords = np.argwhere(region)
            cy_r, cx_r = coords.mean(axis=0)
            ry = (coords[:,0].max() - coords[:,0].min()) / 2
            rx = (coords[:,1].max() - coords[:,1].min()) / 2
            if rx > 0 and ry > 0:
                errors['dust'].append({
                    'x': float(cx_r), 'y': float(cy_r), 'r': float(np.hypot(rx, ry))
                })
    except:
        pass
    
    # Dead zone
    try:
        raw_min = np.nanmin(data)
        raw_max = np.nanmax(data)
        tol = max(1.0, 0.02*(raw_max-raw_min))
        dead_mask = (data <= raw_min + tol).astype(np.uint8)
        labeled, n_obj = ndimage.label(dead_mask)
        if n_obj > 0:
            sizes = ndimage.sum(dead_mask, labeled, index=range(1, n_obj+1))
            if len(sizes):
                errors['dead_zone_fraction'] = float(np.max(sizes)) / float(h*w)
    except:
        pass
    
    # Edge
    try:
        border = max(20, int(min(h,w)*0.05))
        center_med = np.nanmedian(norm[border:h-border, border:w-border])
        center_std = np.nanstd(norm[border:h-border, border:w-border])
        
        edge_strips = [
            norm[:border, :], norm[h-border:, :],
            norm[:, :border], norm[:, w-border:],
        ]
        for strip in edge_strips:
            strip_med = np.nanmedian(strip)
            if abs(strip_med - center_med) > 3*center_std:
                errors['edge'] = True
                break
    except:
        pass
    
    return errors

def annotate_errors(ax, errors):
    """Annotate errors on plot."""
    legend_handles = []
    
    for s in errors['scratches']:
        ax.plot([s['x0'], s['x1']], [s['y0'], s['y1']], 
               color='magenta', linewidth=1.2, alpha=0.8, linestyle='--')
    
    for t in errors['trailing']:
        ellipse = mpatches.Ellipse((t['x'], t['y']), width=t['w'], height=t['h'],
                                   angle=t['angle'], edgecolor='orange', 
                                   facecolor='none', linewidth=1.5, alpha=0.85)
        ax.add_patch(ellipse)
    
    for s in errors['saturation']:
        circ = plt.Circle((s['x'], s['y']), s['r'], edgecolor='red',
                         facecolor='none', linewidth=1.5, alpha=0.8, linestyle='-.')
        ax.add_patch(circ)
    
    for d in errors['dust']:
        circ = plt.Circle((d['x'], d['y']), d['r'], edgecolor='deepskyblue',
                         facecolor='none', linewidth=1.2, alpha=0.8, linestyle=':')
        ax.add_patch(circ)
    
    if errors['edge']:
        ax_h, ax_w = ax.get_ylim(), ax.get_xlim()
        img_h, img_w = abs(ax_h[1]-ax_h[0]), abs(ax_w[1]-ax_w[0])
        rect = mpatches.Rectangle((min(ax_w), min(ax_h)), img_w, img_h,
                                 edgecolor='lime', facecolor='none', linewidth=2.5, alpha=0.7)
        ax.add_patch(rect)

# ============================================================
# PLATE QUALITY ASSESSMENT
# ============================================================
CATEGORY_DISPLAY = {
    'ideal': 'Ideal Image',
    'good_target': 'Good Image, Target',
    'good_no_target': 'Good Image, No Target',
    'defective_target': 'Defective Image, Target',
    'defective_no_target': 'Defective Image, No Target',
}
CATEGORY_RANK = {'ideal': 0, 'good_target': 1, 'good_no_target': 1,
                 'defective_target': 2, 'defective_no_target': 2}

def classify_plate(errors, apass_info, target_found, calibration, 
                   lim_mag_apass=None, lim_mag_atlas=None):
    """Classify plate quality."""
    
    # Defect counts
    n_scratches = len(errors['scratches'])
    n_trailing = len(errors['trailing'])
    n_saturation = len(errors['saturation'])
    n_dust = len(errors['dust'])
    n_edge = 1 if errors['edge'] else 0
    dead_frac = errors.get('dead_zone_fraction', 0.0)
    sat_frac = errors.get('saturation_area_fraction', 0.0)
    
    total_defects = n_scratches + n_trailing + n_saturation + n_dust + n_edge
    
    def defective():
        return 'defective_target' if target_found else 'defective_no_target'
    
    # Hard defect gates
    if dead_frac >= 0.08:
        return defective()
    if sat_frac >= 0.05:
        return defective()
    if total_defects >= 8:
        return defective()
    if n_saturation >= 8:
        return defective()
    if errors['edge'] and total_defects >= 6:
        return defective()
    
    defect_good = total_defects < 8
    
    # APASS match quality
    apass_good = False
    if apass_info:
        n_valid = apass_info.get('n_valid', 0)
        n_total = apass_info.get('n_total', 0)
        rms = apass_info.get('rms')
        slope = apass_info.get('slope')
        
        if n_total > 0:
            frac = n_valid / n_total if n_total > 0 else 0
            match_ok = (frac >= 0.3) or (n_valid >= 15)
            fit_ok = (rms is None) or (rms <= MAX_FIT_RMS)
            slope_ok = (slope is None) or (abs(slope) >= MIN_FIT_SLOPE)
            apass_good = match_ok and fit_ok and slope_ok
    
    # Depth
    lim_mag = lim_mag_apass if lim_mag_apass else lim_mag_atlas
    lim_mag_median = 15.0  # Approximate median
    depth_good = (lim_mag is None) or (lim_mag >= lim_mag_median - 1.0)
    
    confidently_good = defect_good and apass_good and depth_good
    
    if confidently_good:
        return 'ideal' if target_found else 'good_no_target'
    if defect_good:
        return 'good_target' if target_found else 'good_no_target'
    return defective()

def get_plate_category(meta):
    return meta.get('category', 'defective_no_target')

# ============================================================
# PLATE PROCESSING
# ============================================================
def process_plate(fits_path, paradigm_entries=None, version=VERSION):
    """Process a single plate with full pipeline."""
    try:
        # Load data
        data = astrofits.getdata(fits_path).astype(float)
        header = astrofits.getheader(fits_path)
        wcs = WCS(header)
        
        # Get date
        date_obs = 'Unknown'
        for key in ['DATE-OBS', 'DATE', 'DATEOBS', 'MJD-OBS']:
            if key in header:
                date_obs = str(header[key])
                break
        
        # Defect detection
        errors = detect_plate_errors(data)
        
        # Background subtraction
        data_sub, bkg_rms = subtract_background_2d(data)
        
        # Get catalogs
        apass_stars = get_field_catalog(wcs, data.shape, 'apass')
        gaia_stars = get_field_catalog(wcs, data.shape, 'gaia')
        
        # Detect sources
        detections, target_idx, target_tier = detect_sources_robust(
            data_sub, wcs, apass_stars, TARGET_COORD
        )
        
        if detections is None or len(detections['x']) == 0:
            return {
                'path': str(fits_path), 'date': date_obs,
                'target_found': False, 'n_sources': 0,
                'quality': {'category': 'poor', 'score': 0},
                'errors': errors,
                'version': version,
                'category': 'defective_no_target'
            }
        
        # Extract detections
        x, y = detections['x'], detections['y']
        ra, dec = detections['ra'], detections['dec']
        bmag = detections['bmag']
        bmag_err = detections['bmag_err']
        flux = detections['flux']
        bkg_std = detections['bkg_std']
        snr = detections['snr']
        
        # Refine centroids
        x_ref, y_ref = refine_centroids(data_sub, x, y)
        
        # Measure photometry
        fluxes, bkg_med, bkg_std_all = measure_photometry(data_sub, x_ref, y_ref)
        
        # Use default aperture flux
        flux_aper = fluxes[DEFAULT_APERTURE]
        
        # Calculate SNR
        area = np.pi * DEFAULT_APERTURE**2
        snr_calc = flux_aper / (bkg_std_all * np.sqrt(area))
        
        # Fit PSF
        fwhm = fit_psf_fwhm(data_sub, x_ref, y_ref)
        
        # Detect saturation
        is_saturated, max_vals = detect_saturation(data, x_ref, y_ref, fluxes)
        
        # Calculate instrumental magnitudes
        inst_mags = -2.5 * np.log10(np.maximum(flux_aper, 1e-10))
        
        # Calibrate
        calibration = calibrate_photometry(inst_mags, bmag, bmag_err)
        
        # Calculate calibrated magnitudes
        cal_mag = np.full(len(x_ref), np.nan)
        if calibration is not None:
            cal_mag = (inst_mags - calibration['intercept']) / calibration['slope']
        
        # Upper limits for non-detections
        upper_limit = np.full(len(x_ref), np.nan)
        for i in range(len(x_ref)):
            if snr_calc[i] < SIGNIF_THRESHOLD:
                upper_limit[i] = compute_upper_limit(flux_aper[i], bkg_std_all[i], area)
        
        # APASS match info
        valid_apass = np.isfinite(bmag) & np.isfinite(inst_mags) & ~is_saturated
        apass_info = {
            'n_valid': valid_apass.sum(),
            'n_total': len(bmag),
            'rms': calibration['rms'] if calibration else None,
            'slope': calibration['slope'] if calibration else None,
        }
        
        # Target info
        target_found = target_idx is not None
        target_mag = cal_mag[target_idx] if target_found and target_idx is not None else np.nan
        target_snr = snr_calc[target_idx] if target_found and target_idx is not None else 0
        target_sat = is_saturated[target_idx] if target_found and target_idx is not None else False
        
        # Plate limit
        lim_mag_apass, lim_mag_atlas = get_plate_limits(fits_path)
        plate_limit = compute_plate_limit(bkg_std_all, calibration)
        
        # Paradigm labels
        paradigm_labels = []
        if paradigm_entries:
            paradigm_labels = match_paradigm(ra, dec, paradigm_entries)
        
        # Classify
        category = classify_plate(errors, apass_info, target_found, 
                                  calibration, lim_mag_apass, lim_mag_atlas)
        
        # Quality score
        score = 0
        if target_found:
            score += 30
        if calibration and calibration['rms'] < 0.5:
            score += 30
        if len(x) > 20:
            score += 20
        if errors['dead_zone_fraction'] < 0.01:
            score += 10
        if errors['saturation_area_fraction'] < 0.01:
            score += 10
        
        return {
            'path': str(fits_path),
            'date': date_obs,
            'target_found': target_found,
            'target_mag': target_mag,
            'target_snr': target_snr,
            'target_saturated': target_sat,
            'target_tier': target_tier,
            'target_idx': target_idx,
            'n_sources': len(x_ref),
            'x': x_ref,
            'y': y_ref,
            'ra': ra,
            'dec': dec,
            'bmag': bmag,
            'bmag_err': bmag_err,
            'inst_mag': inst_mags,
            'cal_mag': cal_mag,
            'flux': flux_aper,
            'snr': snr_calc,
            'fwhm': fwhm,
            'upper_limit': upper_limit,
            'is_saturated': is_saturated,
            'paradigm_labels': paradigm_labels,
            'calibration': calibration,
            'apass_info': apass_info,
            'plate_limit': plate_limit,
            'errors': errors,
            'category': category,
            'quality_score': score,
            'version': version
        }
        
    except Exception as e:
        print(f"Error processing {fits_path.name}: {e}")
        return None

def match_paradigm(ra, dec, paradigm_entries, tolerance=PARADIGM_MATCH_ARCSEC):
    """Match sources to paradigm labels."""
    if not paradigm_entries:
        return [''] * len(ra)
    
    # Convert to lists if needed
    entries_list = list(paradigm_entries.values()) if isinstance(paradigm_entries, dict) else paradigm_entries
    
    para_coords = SkyCoord([e['ra'] for e in entries_list], 
                          [e['dec'] for e in entries_list], unit='deg')
    src_coords = SkyCoord(ra*u.deg, dec*u.deg)
    
    idx, sep, _ = match_coordinates_sky(src_coords, para_coords)
    
    labels = []
    for i in range(len(ra)):
        if sep[i].arcsec <= tolerance:
            labels.append(entries_list[idx[i]]['label'])
        else:
            labels.append('')
    return labels

# ============================================================
# LIGHTCURVE
# ============================================================
def build_lightcurve(plate_db, exclude_defective=True):
    """Build lightcurve from processed plates."""
    rows = []
    
    for f, data in plate_db.items():
        if data is None:
            continue
        
        # Parse date
        try:
            date = Time(data['date'])
            jd = date.jd
        except:
            continue
        
        # Quality filter
        if exclude_defective and data.get('category') in ['defective_target', 'defective_no_target']:
            continue
        
        rows.append({
            'plate': Path(f).name,
            'jd': jd,
            'date': data.get('date', 'Unknown'),
            'category': data.get('category', 'unknown'),
            'target_found': data.get('target_found', False),
            'target_mag': data.get('target_mag', np.nan),
            'target_snr': data.get('target_snr', 0),
            'target_saturated': data.get('target_saturated', False),
            'n_sources': data.get('n_sources', 0),
            'quality_score': data.get('quality_score', 0),
            'plate_limit': data.get('plate_limit', np.nan)
        })
    
    df = pd.DataFrame(rows)
    if len(df) > 0:
        df = df.sort_values('jd').reset_index(drop=True)
    return df

def plot_lightcurve(df, show_limits=True, save_path=None):
    """Plot lightcurve with proper error bars."""
    if len(df) == 0:
        print("No data to plot")
        return
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                   gridspec_kw={'height_ratios': [3, 1]})
    
    # Main plot
    detected = df[df['target_found']]
    nondetected = df[~df['target_found']]
    
    if len(detected) > 0:
        # Separate saturated
        sat = detected[detected['target_saturated']]
        unsat = detected[~detected['target_saturated']]
        
        if len(unsat) > 0:
            ax1.errorbar(unsat['jd'], unsat['target_mag'], 
                        yerr=0.3, fmt='o', color='blue', 
                        markersize=4, capsize=3, label='Detected')
        
        if len(sat) > 0:
            ax1.errorbar(sat['jd'], sat['target_mag'], 
                        yerr=0.5, fmt='^', color='orange',
                        markersize=5, capsize=3, label='Saturated')
    
    if show_limits and len(nondetected) > 0:
        # Use 3-sigma limits for non-detections
        ax1.scatter(nondetected['jd'], nondetected['plate_limit'], 
                   marker='v', s=20, color='gray', alpha=0.5,
                   label='Non-detection (3σ)')
    
    ax1.invert_yaxis()
    ax1.set_xlabel('Julian Date')
    ax1.set_ylabel('B Magnitude')
    ax1.set_title(f'{TARGET_NAME} Lightcurve')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Quality plot
    ax2.scatter(df['jd'], df['quality_score'], 
               c=df['quality_score'], s=20, cmap='viridis')
    ax2.set_xlabel('Julian Date')
    ax2.set_ylabel('Quality Score')
    ax2.set_ylim(0, 100)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to {save_path}")

# ============================================================
# REVIEW SYSTEM (Preserved from original)
# ============================================================
_review_cluster_of = {}
_review_cluster_members = {}

def _plate_signature(meta):
    """Create signature for clustering."""
    if meta is None:
        return (0, 0, 0, 0, 0)
    
    errs = meta.get('errors', {})
    total_defects = (
        len(errs.get('scratches', [])) + len(errs.get('trailing', [])) +
        len(errs.get('saturation', [])) + len(errs.get('dust', [])) +
        (1 if errs.get('edge') else 0)
    )
    dead_bucket = min(int(errs.get('dead_zone_fraction', 0.0) * 20), 3)
    sat_bucket = min(int(errs.get('saturation_area_fraction', 0.0) * 20), 3)
    n_bucket = min(meta.get('n_sources', 0) // 10, 10)
    apass_info = meta.get('apass_info', {})
    n_valid = apass_info.get('n_valid', 0)
    match_bucket = min(n_valid // 5, 6)
    return (min(total_defects, 8), dead_bucket, sat_bucket, n_bucket, match_bucket)

def _cluster_plates_by_category(max_total=REVIEW_MAX_TOTAL):
    """Cluster plates for review."""
    global _review_cluster_of, _review_cluster_members
    _review_cluster_of = {}
    _review_cluster_members = {}
    
    by_category = {}
    for f_str, meta in plate_db.items():
        if meta is None or meta.get('reviewed', False):
            continue
        cat = get_plate_category(meta)
        by_category.setdefault(cat, []).append((f_str, meta))
    
    ranked_by_category = {}
    for cat in CATEGORY_DISPLAY.keys():
        items = by_category.get(cat, [])
        if not items:
            continue
        groups = {}
        for f_str, meta in items:
            sig = _plate_signature(meta)
            groups.setdefault(sig, []).append(f_str)
        ranked_by_category[cat] = sorted(groups.items(), key=lambda kv: -len(kv[1]))
    
    representatives = []
    cluster_id = 0
    cursors = {cat: 0 for cat in ranked_by_category}
    
    while len(representatives) < max_total:
        progressed = False
        for cat in ranked_by_category:
            if len(representatives) >= max_total:
                break
            idx = cursors[cat]
            if idx >= len(ranked_by_category[cat]):
                continue
            sig, member_files = ranked_by_category[cat][idx]
            cursors[cat] += 1
            progressed = True
            
            for mf in member_files:
                _review_cluster_of[mf] = cluster_id
            _review_cluster_members[cluster_id] = member_files
            member_files_sorted = sorted(member_files, key=lambda fs: plate_db[fs].get('n_sources', 0))
            rep = member_files_sorted[len(member_files_sorted) // 2]
            representatives.append(rep)
            cluster_id += 1
        if not progressed:
            break
    
    # Pad with leftovers if needed
    if len(representatives) < max_total:
        already_covered = set()
        for members in _review_cluster_members.values():
            already_covered.update(members)
        for f_str, meta in plate_db.items():
            if len(representatives) >= max_total:
                break
            if meta is None or meta.get('reviewed', False):
                continue
            if f_str in already_covered:
                continue
            _review_cluster_of[f_str] = cluster_id
            _review_cluster_members[cluster_id] = [f_str]
            representatives.append(f_str)
            already_covered.add(f_str)
            cluster_id += 1
    
    return representatives

def _pending_review_plates(representatives):
    """Get pending review plates."""
    items = []
    for f_str in representatives:
        meta = plate_db.get(f_str)
        if meta is None or meta.get('reviewed', False):
            continue
        f = Path(f_str)
        cluster_id = _review_cluster_of.get(f_str)
        n_members = len(_review_cluster_members.get(cluster_id, [f_str]))
        cat = get_plate_category(meta)
        label = (f"{f.name} | {CATEGORY_DISPLAY[cat]} | "
                f"represents {n_members} plate(s) | "
                f"score: {meta.get('quality_score', 0)}")
        items.append((n_members, label, f))
    items.sort(key=lambda t: -t[0])
    return [(label, f) for _, label, f in items]

# ============================================================
# INTERACTIVE UI
# ============================================================
# Scan controls
scan_progress = widgets.IntProgress(value=0, min=0, max=len(cutouts))
scan_status = widgets.Label(value="Not scanned yet")
scan_progress_html = widgets.HTML(value="")
run_initial_btn = widgets.Button(description="Run Initial Scan", button_style='info')
recompute_btn = widgets.Button(description="Recompute All", button_style='warning')
scan_warning = widgets.HTML(
    "<small>Initial Scan queries catalogs for each plate. "
    "Recompute All re-derives calibration from cached data.</small>"
)

# Filter controls
category_filter = widgets.Dropdown(description="Filter:")
search_box = widgets.Text(description="Search:", placeholder="filter by filename")
main_category_filter = widgets.Dropdown(description="Filter:")
main_search_box = widgets.Text(description="Search:", placeholder="filter by filename")

# Review panel
review_plate_dropdown = widgets.Dropdown(description="Review:")
review_status = widgets.Label(value="")
approve_btn = widgets.Button(description="Approve", button_style='success')
reject_btn = widgets.Button(description="Reject", button_style='danger')
review_instructions = widgets.HTML(
    f"<small>Review up to {REVIEW_MAX_TOTAL} representative plates. "
    "<b>Approve</b> confirms the category. "
    "<b>Reject</b> flags it as wrong (doesn't change category).</small>"
)

# Paradigm panel
paradigm_plate_dropdown = widgets.Dropdown(description="Paradigm plate:")
paradigm_status = widgets.Label(value="")
paradigm_instructions = widgets.HTML(
    "<small>Select a plate, then click 'Add Selected' to use it as paradigm. "
    "This labels all bright sources for matching across plates.</small>"
)
paradigm_add_btn = widgets.Button(description="Add Selected", button_style='info')
paradigm_clear_btn = widgets.Button(description="Clear All", button_style='danger')
apply_paradigm_btn = widgets.Button(description="Apply Paradigm", button_style='success')
paradigm_output = widgets.Output()
paradigm_table_output = widgets.Output()

# Main viewer
dropdown = widgets.Dropdown(description="Plate:")
dropdown_status = widgets.Label(value="")
display_mode = widgets.ToggleButtons(options=["GAIA ID", "Object ID"], value="GAIA ID", description="Labels:")
error_toggle = widgets.ToggleButtons(options=["Show Errors", "Hide Errors"], value="Show Errors", description="Errors:")

# Lightcurve controls
lightcurve_btn = widgets.Button(description="Generate Lightcurve", button_style='info')
lightcurve_save = widgets.Checkbox(value=False, description="Save CSV")
lightcurve_exclude = widgets.Checkbox(value=True, description="Exclude Defective")
lightcurve_show_limits = widgets.Checkbox(value=True, description="Show Upper Limits")
lightcurve_output = widgets.Output()

# Shared plot output
shared_plot_output = widgets.Output()
shared_info_output = widgets.Output()
shared_diagnostics_output = widgets.Output()

# ============================================================
# VIEWER STATE
# ============================================================
_viewer_state = {
    'fig': None, 'ax': None, 'cids': [],
    'fits_path': None, 'mode': None,
    'xs': [], 'ys': [], 'manual': [],
    'wcs': None, 'data': None,
    'highlight_marker': None, 'annotation': None,
    'star_scatter': None, 'star_meta': [],
}

# ============================================================
# VIEWER FUNCTIONS (Preserved from original)
# ============================================================
def _teardown_shared_plot():
    """Clean up shared plot."""
    fig = _viewer_state.get('fig')
    if fig is not None:
        for cid in _viewer_state.get('cids', []):
            try:
                fig.canvas.mpl_disconnect(cid)
            except:
                pass
        plt.close(fig)
    _viewer_state.update({'fig': None, 'ax': None, 'cids': [],
                          'highlight_marker': None, 'annotation': None})

def _star_label_text(i):
    """Get star label text."""
    meta = _viewer_state['star_meta'][i]
    lines = [f"★ {TARGET_NAME}" if meta.get('is_target') else meta.get('name', 'Unknown')]
    if np.isfinite(meta.get('bmag', np.nan)):
        lines.append(f"APASS B={meta['bmag']:.2f}")
    if np.isfinite(meta.get('inst_mag', np.nan)):
        lines.append(f"inst={meta['inst_mag']:.2f}")
    if np.isfinite(meta.get('cal_mag', np.nan)):
        lines.append(f"B={meta['cal_mag']:.2f}")
    if meta.get('is_saturated'):
        lines.append("SATURATED")
    return "\n".join(lines)

def _highlight_star(idx):
    """Highlight a star."""
    ax = _viewer_state['ax']
    if ax is None or idx is None or idx >= len(_viewer_state['xs']):
        return
    
    x, y = _viewer_state['xs'][idx], _viewer_state['ys'][idx]
    hl = _viewer_state['highlight_marker']
    hl.set_data([x], [y])
    hl.set_visible(True)
    
    ann = _viewer_state['annotation']
    ann.xy = (x, y)
    ann.set_text(_star_label_text(idx))
    ann.set_visible(True)
    
    _viewer_state['fig'].canvas.draw_idle()
    
    with shared_info_output:
        clear_output(wait=True)
        print(_star_label_text(idx).replace('\n', '   |   '))

def render_plate_view(fits_path, mode='readonly'):
    """Render plate view."""
    _teardown_shared_plot()
    
    meta = plate_db.get(str(fits_path))
    if meta is None:
        with shared_plot_output:
            clear_output()
            print("No data available")
        return
    
    # Load image
    data = astrofits.getdata(fits_path)
    header = astrofits.getheader(fits_path)
    wcs = WCS(header)
    
    # Get data
    n = meta.get('n_sources', 0)
    x = meta.get('x', np.array([]))
    y = meta.get('y', np.array([]))
    snr = meta.get('snr', np.array([]))
    cal_mag = meta.get('cal_mag', np.array([]))
    bmag = meta.get('bmag', np.array([]))
    inst_mag = meta.get('inst_mag', np.array([]))
    is_saturated = meta.get('is_saturated', np.array([]))
    fwhm = meta.get('fwhm', np.array([]))
    target_idx = meta.get('target_idx')
    paradigm_labels = meta.get('paradigm_labels', [])
    
    with shared_plot_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 10))
        
        # Display image
        vmin = np.percentile(data, 5)
        vmax = np.percentile(data, 99.5)
        ax.imshow(data, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
        
        # Plot sources
        if n > 0 and len(x) > 0:
            # Color by SNR
            colors = plt.cm.viridis(snr / max(snr.max(), 1)) if snr.max() > 0 else 'blue'
            sc = ax.scatter(x, y, c=colors, s=30, alpha=0.7, picker=True, pickradius=8)
            
            # Highlight target
            if target_idx is not None and target_idx < len(x):
                ax.scatter([x[target_idx]], [y[target_idx]], 
                          s=150, edgecolors='cyan', facecolors='none', linewidths=2)
                ax.text(x[target_idx], y[target_idx]+10, '★', color='cyan', 
                       fontsize=12, ha='center', va='bottom')
            
            # Label paradigm matches
            for i, label in enumerate(paradigm_labels):
                if label and i < len(x):
                    ax.annotate(label, (x[i], y[i]), fontsize=6, color='white',
                               bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.7))
        
        # Show defects
        if error_toggle.value == 'Show Errors':
            annotate_errors(ax, meta.get('errors', {}))
        
        # Plate info
        title = f"{fits_path.name} | {CATEGORY_DISPLAY.get(meta.get('category', 'unknown'), meta.get('category', 'unknown'))} | {n} sources"
        if meta.get('target_found', False):
            title += f" | Target: {meta.get('target_mag', np.nan):.2f} mag"
        ax.set_title(title, fontsize=10)
        
        # Highlight marker
        highlight_marker, = ax.plot([], [], 'o', markersize=24, 
                                    markerfacecolor='none', markeredgecolor='#FFD700',
                                    markeredgewidth=3, zorder=5)
        
        annotation = ax.annotate('', xy=(0,0), xytext=(15,15), textcoords='offset points',
                                 color='black', fontsize=9,
                                 bbox=dict(boxstyle='round', fc='yellow', alpha=0.9),
                                 arrowprops=dict(arrowstyle='->'))
        annotation.set_visible(False)
        
        _viewer_state.update({
            'fig': fig, 'ax': ax, 'fits_path': fits_path, 'mode': mode,
            'xs': x.tolist() if len(x) > 0 else [], 
            'ys': y.tolist() if len(y) > 0 else [],
            'wcs': wcs, 'data': data,
            'highlight_marker': highlight_marker, 'annotation': annotation,
            'star_scatter': sc if n > 0 else None,
            'star_meta': [],
            'cids': []
        })
        
        # Setup picker
        cids = []
        if n > 0 and hasattr(sc, 'figure'):
            def on_pick(event):
                if event.artist != _viewer_state['star_scatter']:
                    return
                _highlight_star(event.ind[0])
            cids.append(fig.canvas.mpl_connect('pick_event', on_pick))
        _viewer_state['cids'] = cids
        
        plt.tight_layout()
        plt.show()
    
    # Show info
    with shared_info_output:
        clear_output()
        if n > 0:
            print(f"{n} sources shown. Click a star for details.")
            if meta.get('target_found', False):
                print(f"Target: mag={meta.get('target_mag', np.nan):.2f}, SNR={meta.get('target_snr', 0):.1f}")
            if meta.get('calibration'):
                print(f"Calibration: slope={meta['calibration']['slope']:.3f}, RMS={meta['calibration']['rms']:.3f}")

# ============================================================
# UI CALLBACKS
# ============================================================
def format_eta(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return "calculating..."
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"

def render_scan_progress(n_done, total, start_time, status_word="Scanning"):
    """Render scan progress."""
    elapsed = time.monotonic() - start_time
    pct = (n_done / total * 100) if total else 0
    rate = (n_done / elapsed) if elapsed > 0 and n_done > 0 else 0
    remaining = ((total - n_done) / rate) if rate > 0 else None
    bar_width = 380
    filled = int(bar_width * pct / 100)
    color = '#4CAF50'
    scan_progress_html.value = f"""
    <div style="font-family: monospace; font-size: 13px; line-height: 1.5;">
      <div style="display:flex; align-items:center;">
        <div style="width:{bar_width}px; height:16px; background:#333;
                    border-radius:4px; overflow:hidden; margin-right:10px;">
          <div style="width:{filled}px; height:100%; background:{color};
                      transition: width 0.3s;"></div>
        </div>
        <b>{pct:5.1f}%</b>
      </div>
      <div style="margin-top:4px;">
        {status_word} &nbsp;
        <b>{n_done:,} / {total:,}</b> plates &nbsp;|&nbsp;
        <b>{rate:.2f}</b> plates/sec &nbsp;|&nbsp;
        Elapsed <b>{format_eta(elapsed)}</b> &nbsp;|&nbsp;
        ETA <b>{format_eta(remaining)}</b>
      </div>
    </div>
    """

def run_initial_scan(plates=None):
    """Run initial scan of all plates."""
    target_plates = plates if plates is not None else cutouts
    total = max(len(target_plates), 1)
    scan_progress.max = total
    scan_progress.value = 0
    start_time = time.monotonic()
    _tick_state = {'last_render': 0.0}
    
    def tick(n_done):
        scan_progress.value = n_done
        now = time.monotonic()
        if now - _tick_state['last_render'] > 0.15 or n_done == total:
            render_scan_progress(n_done, total, start_time, status_word="Initial scan")
            _tick_state['last_render'] = now
    
    for i, f in enumerate(target_plates):
        cached = plate_db.get(str(f))
        if cached and cached.get('version', 0) == VERSION:
            tick(i+1)
            continue
        
        scan_status.value = f"Processing: {f.name}"
        try:
            result = process_plate(f)
            if result:
                plate_db[str(f)] = result
                save_cache('plate_db', plate_db)
        except Exception as e:
            print(f"Error: {f.name}: {e}")
        
        if (i+1) % 10 == 0:
            print(f"Processed {i+1}/{total} plates")
        
        tick(i+1)
    
    render_scan_progress(total, total, start_time, status_word="Scan complete")
    scan_status.value = f"Complete: {len(plate_db)} plates processed"
    
    refresh_filter_options()
    rebuild_dropdown()
    refresh_review_queue()
    update_paradigm_dropdown()

def recompute_all():
    """Recompute calibration from cached data."""
    updated = 0
    for f_str, meta in plate_db.items():
        if meta is None:
            continue
        
        # Recompute calibration
        if 'inst_mag' in meta and 'bmag' in meta:
            cal = calibrate_photometry(meta['inst_mag'], meta['bmag'], 
                                      np.full_like(meta['bmag'], 0.1))
            if cal:
                meta['calibration'] = cal
                if 'cal_mag' in meta and len(meta['cal_mag']) > 0:
                    meta['cal_mag'] = (meta['inst_mag'] - cal['intercept']) / cal['slope']
                
                # Recompute apass info
                valid = np.isfinite(meta['bmag']) & np.isfinite(meta['inst_mag'])
                meta['apass_info'] = {
                    'n_valid': valid.sum(),
                    'n_total': len(meta['bmag']),
                    'rms': cal['rms'],
                    'slope': cal['slope']
                }
        
        # Recompute category
        if 'apass_info' in meta and 'errors' in meta:
            lim_mag_apass, lim_mag_atlas = get_plate_limits(Path(f_str))
            meta['category'] = classify_plate(
                meta['errors'], meta['apass_info'], 
                meta.get('target_found', False),
                meta.get('calibration'),
                lim_mag_apass, lim_mag_atlas
            )
            updated += 1
    
    save_cache('plate_db', plate_db)
    refresh_filter_options()
    rebuild_dropdown()
    refresh_review_queue()
    print(f"Recomputed {updated} plates")

def refresh_review_queue():
    """Refresh review queue."""
    reps = _cluster_plates_by_category()
    pending = _pending_review_plates(reps)
    review_plate_dropdown.options = pending
    review_status.value = f"{len(pending)} plates awaiting review"

def refresh_filter_options():
    """Refresh filter dropdowns."""
    counts = {'ideal': 0, 'good_target': 0, 'good_no_target': 0,
              'defective_target': 0, 'defective_no_target': 0}
    total = len(plate_db)
    
    for meta in plate_db.values():
        if meta:
            cat = get_plate_category(meta)
            counts[cat] = counts.get(cat, 0) + 1
    
    options = [
        (f"all ({total})", "all"),
        (f"{CATEGORY_DISPLAY['ideal']} ({counts['ideal']})", "ideal"),
        (f"{CATEGORY_DISPLAY['good_target']} ({counts['good_target']})", "good_target"),
        (f"{CATEGORY_DISPLAY['good_no_target']} ({counts['good_no_target']})", "good_no_target"),
        (f"{CATEGORY_DISPLAY['defective_target']} ({counts['defective_target']})", "defective_target"),
        (f"{CATEGORY_DISPLAY['defective_no_target']} ({counts['defective_no_target']})", "defective_no_target"),
    ]
    
    for dd in (category_filter, main_category_filter):
        cur = dd.value
        dd.options = options
        if cur in [o[1] for o in options]:
            dd.value = cur

def rebuild_dropdown():
    """Rebuild main dropdown."""
    if not plate_db:
        dropdown.options = []
        dropdown_status.value = "Run initial scan first"
        return
    
    options = []
    for f_str, meta in plate_db.items():
        if meta is None:
            continue
        f = Path(f_str)
        cat = get_plate_category(meta)
        label = f"{f.name} | {CATEGORY_DISPLAY.get(cat, cat)} | {meta.get('n_sources', 0)} sources"
        options.append((label, f))
    
    options = options[:MAX_DROPDOWN_OPTIONS]
    dropdown.options = options
    dropdown_status.value = f"{len(plate_db)} plates available"

def update_paradigm_dropdown():
    """Update paradigm dropdown."""
    if paradigm_db:
        options = []
        for f_str in paradigm_db.keys():
            if f_str in plate_db:
                f = Path(f_str)
                meta = plate_db[f_str]
                if meta:
                    options.append((f"{f.name} ({meta.get('n_sources', 0)} sources)", Path(f_str)))
        paradigm_plate_dropdown.options = options
        if options:
            paradigm_plate_dropdown.value = options[0][1]

def approve_plate(b):
    """Approve current review plate."""
    if not review_plate_dropdown.value:
        return
    
    f_str = str(review_plate_dropdown.value)
    cluster_id = _review_cluster_of.get(f_str)
    members = _review_cluster_members.get(cluster_id, [f_str])
    
    for m in members:
        if m in plate_db:
            plate_db[m]['reviewed'] = True
            plate_db[m]['approved'] = True
    
    save_cache('plate_db', plate_db)
    refresh_review_queue()
    print(f"Approved {len(members)} plate(s)")

def reject_plate(b):
    """Reject current review plate."""
    if not review_plate_dropdown.value:
        return
    
    f_str = str(review_plate_dropdown.value)
    cluster_id = _review_cluster_of.get(f_str)
    members = _review_cluster_members.get(cluster_id, [f_str])
    
    for m in members:
        if m in plate_db:
            plate_db[m]['reviewed'] = True
            plate_db[m]['approved'] = False
    
    save_cache('plate_db', plate_db)
    refresh_review_queue()
    print(f"Rejected {len(members)} plate(s)")

def add_paradigm(b):
    """Add selected plate as paradigm."""
    if not paradigm_plate_dropdown.value:
        return
    
    f_str = str(paradigm_plate_dropdown.value)
    meta = plate_db.get(f_str)
    if not meta or meta.get('n_sources', 0) == 0:
        with paradigm_output:
            clear_output()
            print("No sources on this plate")
        return
    
    # Add bright, high SNR sources
    entries = {}
    for i in range(meta['n_sources']):
        if meta['snr'][i] > 10 and meta['cal_mag'][i] < 15:
            entries[f"{i}"] = {
                'ra': meta['ra'][i],
                'dec': meta['dec'][i],
                'label': f"Ref_{i:03d}",
                'mag': meta['cal_mag'][i],
                'snr': meta['snr'][i]
            }
    
    paradigm_db[f_str] = entries
    save_cache('paradigm_db', paradigm_db)
    
    with paradigm_output:
        clear_output()
        print(f"Added {len(entries)} paradigm references from {Path(f_str).name}")

def clear_paradigm(b):
    """Clear all paradigm labels."""
    paradigm_db.clear()
    save_cache('paradigm_db', paradigm_db)
    with paradigm_output:
        clear_output()
        print("Cleared all paradigm labels")

def apply_paradigm(b):
    """Apply paradigm labels to all plates."""
    if not paradigm_db:
        paradigm_status.value = "No paradigm labels to apply"
        return
    
    paradigm_status.value = "Applying paradigm..."
    
    # Get all paradigm entries
    all_entries = []
    for entries in paradigm_db.values():
        all_entries.extend(entries.values())
    
    if not all_entries:
        return
    
    # Re-process plates with paradigm
    for f_str, meta in plate_db.items():
        if meta is None or meta.get('n_sources', 0) == 0:
            continue
        
        # Match paradigm labels
        labels = match_paradigm(meta['ra'], meta['dec'], all_entries)
        meta['paradigm_labels'] = labels
    
    save_cache('plate_db', plate_db)
    paradigm_status.value = "Paradigm applied"
    print(f"Applied paradigm to {len(plate_db)} plates")

def on_review_select(change):
    """Handle review selection."""
    if change['new']:
        render_plate_view(change['new'], 'readonly')

def on_main_select(change):
    """Handle main viewer selection."""
    if change['new']:
        render_plate_view(change['new'], 'readonly')

def on_paradigm_select(change):
    """Handle paradigm selection."""
    if change['new']:
        render_plate_view(change['new'], 'readonly')

def generate_lightcurve(b):
    """Generate and display lightcurve."""
    exclude_defective = lightcurve_exclude.value
    show_limits = lightcurve_show_limits.value
    save_csv = lightcurve_save.value
    
    df = build_lightcurve(plate_db, exclude_defective)
    
    with lightcurve_output:
        clear_output()
        if len(df) == 0:
            print("No data to plot")
            return
        
        print(f"Lightcurve: {len(df)} plate(s)")
        print(f"  Detections: {df['target_found'].sum()}")
        print(f"  Non-detections: {(~df['target_found']).sum()}")
        
        plot_lightcurve(df, show_limits=show_limits)
        
        if save_csv:
            csv_path = cache_dir / 'lightcurve.csv'
            df.to_csv(csv_path, index=False)
            print(f"Saved to {csv_path}")

# ============================================================
# WIRE UP CALLBACKS
# ============================================================
run_initial_btn.on_click(lambda b: run_initial_scan())
recompute_btn.on_click(lambda b: recompute_all())
approve_btn.on_click(approve_plate)
reject_btn.on_click(reject_plate)
paradigm_add_btn.on_click(add_paradigm)
paradigm_clear_btn.on_click(clear_paradigm)
apply_paradigm_btn.on_click(apply_paradigm)
lightcurve_btn.on_click(generate_lightcurve)

review_plate_dropdown.observe(on_review_select, names='value')
dropdown.observe(on_main_select, names='value')
paradigm_plate_dropdown.observe(on_paradigm_select, names='value')

category_filter.observe(lambda c: update_paradigm_dropdown(), names='value')
search_box.observe(lambda c: update_paradigm_dropdown(), names='value')
main_category_filter.observe(lambda c: rebuild_dropdown(), names='value')
main_search_box.observe(lambda c: rebuild_dropdown(), names='value')
display_mode.observe(lambda c: render_plate_view(_viewer_state['fits_path'], 'readonly') if _viewer_state.get('fits_path') else None, names='value')
error_toggle.observe(lambda c: render_plate_view(_viewer_state.get('fits_path'), 'readonly') if _viewer_state.get('fits_path') else None, names='value')

# ============================================================
# BUILD UI
# ============================================================
# Scan section
scan_section = widgets.VBox([
    widgets.HTML("<b>Initial Scan</b>"),
    scan_warning,
    widgets.HBox([run_initial_btn, recompute_btn]),
    scan_progress_html,
    scan_status,
])

# Review section
review_section = widgets.VBox([
    widgets.HTML("<b>Review Plates</b>"),
    review_instructions,
    category_filter,
    search_box,
    review_plate_dropdown,
    review_status,
    widgets.HBox([approve_btn, reject_btn]),
])

# Paradigm section
paradigm_section = widgets.VBox([
    widgets.HTML("<b>Paradigm Labels</b>"),
    paradigm_instructions,
    paradigm_plate_dropdown,
    widgets.HBox([paradigm_add_btn, paradigm_clear_btn, apply_paradigm_btn]),
    paradigm_status,
    paradigm_output,
    paradigm_table_output,
])

# Main viewer section
main_section = widgets.VBox([
    widgets.HTML("<b>Plate Viewer</b>"),
    main_category_filter,
    main_search_box,
    dropdown,
    dropdown_status,
    widgets.HBox([display_mode, error_toggle]),
    shared_plot_output,
    shared_info_output,
])

# Lightcurve section
lightcurve_section = widgets.VBox([
    widgets.HTML("<b>Lightcurve</b>"),
    widgets.HBox([lightcurve_btn, lightcurve_save, lightcurve_exclude, lightcurve_show_limits]),
    lightcurve_output,
])

# ============================================================
# DISPLAY UI
# ============================================================
display(widgets.HTML("<h1>V CrA Photometry Pipeline</h1>"))
display(scan_section)
display(review_section)
display(paradigm_section)
display(main_section)
display(lightcurve_section)

# ============================================================
# INITIALIZE UI
# ============================================================
refresh_filter_options()
rebuild_dropdown()
refresh_review_queue()
update_paradigm_dropdown()

print("\n" + "="*60)
print("PIPELINE INITIALIZED")
print("="*60)
print(f"Loaded {len(plate_db)} cached plates")
print(f"Review: {len(_review_cluster_members)} clusters")
print(f"Paradigm: {len(paradigm_db)} plates labeled")
print("\nWorkflow:")
print("1. Run Initial Scan (or use cached data)")
print("2. Review plates (Approve/Reject)")
print("3. Select a Paradigm plate and click 'Add Selected'")
print("4. Click 'Apply Paradigm' to propagate labels")
print("5. Generate Lightcurve")

Found 5402 cutouts
Loaded 0 cached plates
Loaded manifest: 7218 rows
Loaded limits for 2797 of 5402 plates


HTML(value='<h1>V CrA Photometry Pipeline</h1>')


PIPELINE INITIALIZED
Loaded 0 cached plates
Review: 0 clusters
Paradigm: 0 plates labeled

Workflow:
1. Run Initial Scan (or use cached data)
2. Review plates (Approve/Reject)
3. Select a Paradigm plate and click 'Add Selected'
4. Click 'Apply Paradigm' to propagate labels
5. Generate Lightcurve
